# S6E10 — 17 RealMLP-rich + teachers + missing features (fold-0 test)

> **Pipeline position:** 16 teachers → **17 RealMLP-rich + teachers / new keys** → 11 stack

| | |
|---|---|
| **Input** | `X_rich_*` (09), `X_teach_*` (16), `y_train.parquet`; reference checkpoint of `12`; OOF files of the stack members |
| **Output** | `preds/realmlp_rich_teach{A,B,AB}_folds/` (fold-0 checkpoints) |
| **Runtime** | ≈ 2 min per variant on an RTX 3070 Ti |
| **Result** | see *Results & decisions* |

**Three variants, each changing only the input versus the paired reference** (RealMLP-rich, recipe of `10`, 4 networks, seed 0, GPU: fold 0 = 0.96138):

| Variant | Added | Idea |
|---|---|---|
| **A** | `og_xgb_logit`, `og_realmlp_logit` (16) | two more opinions from models trained on the original survey |
| **B** | categorical twins of the two delay columns; in-fold TE of `Class × Travel × Gender` and `Distance × Time-convenient` | the parts of the public recipe not yet in `10` (twins for route, age and the ratings already exist) |
| **A+B** | both | |

**Judged by:** own fold-0 AUC (bar +0.0002) and the fold-0 mini-stack of `15` (10 members + the new run). The teacher logits are also tried directly as extra stack members (they are label-free features, so their fold-0 values are not fitted on any label of those rows).

> **Glossary:** *Categorical twin* = a numeric column also given as a category, so the net can learn a separate effect per value. *In-fold TE* = target encoding fitted on the fold's training part only.

In [1]:
import torch          # first import: on the Mac the kernel crashes if torch comes after numpy/sklearn (OpenMP conflict)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np, pandas as pd, time, gc, warnings
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from sklearn.linear_model import LogisticRegression
from pytabkit import RealMLP_TD_Classifier
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_ENS, SEED = 4, 0
VARIANTS = ['A', 'B', 'AB']
CANDS = [Path('data/processed'), Path('../data/processed')]
PROC_DIR = next(p for p in CANDS if (p / 'X_teach_train.parquet').exists())
PREDS_DIR = next(p for p in [Path('preds'), Path('../preds')] if p.exists())
X0 = pd.read_parquet(PROC_DIR / 'X_rich_train.parquet'); X0_test = pd.read_parquet(PROC_DIR / 'X_rich_test.parquet')
TCH = pd.read_parquet(PROC_DIR / 'X_teach_train.parquet'); TCH_test = pd.read_parquet(PROC_DIR / 'X_teach_test.parquet')
y = pd.read_parquet(PROC_DIR / 'y_train.parquet')['y'].astype(int).values
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X0, y))
tr0, va0 = FOLDS[0]
print(DEVICE, X0.shape, TCH.shape)

C:\Users\89671\anaconda3\envs\kaggle-playground\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


W1010 04:58:19.926000 23528 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


cuda (699635, 92) (699635, 2)


## 1. Building the input of each variant
New TE keys are integer ids (label-free); they are encoded **inside the fold** by `add_te`, like the existing keys. The delay twins: departure delay as stored, arrival delay recovered as `departure − delay_recovered` (the 292 missing arrivals were filled with the departure delay in `03`).

In [2]:
RATINGS = ['inflight_wifi_service','departure/arrival_time_convenient','ease_of_online_booking','gate_location','food_and_drink',
           'online_boarding','seat_comfort','inflight_entertainment','on-board_service','leg_room_service','baggage_handling',
           'checkin_service','cleanliness']
BASE_TWINS = {'route_id': 'flight_distance', 'age_cat': 'age', **{f'{r}_cat': r for r in RATINGS}}
BASE_TE = ['flight_distance', 'age', 'fd_div10', 'fd_mod1000', 'key_age_class', 'key_fd_class', 'key_age_travel']

def build(variant):
    X, Xt = X0.copy(), X0_test.copy()
    twins, te_src = dict(BASE_TWINS), list(BASE_TE)
    if 'A' in variant:
        X = pd.concat([X, TCH], axis=1); Xt = pd.concat([Xt, TCH_test], axis=1)
    if 'B' in variant:
        for D in (X, Xt):
            D['arr_delay_int'] = (D['departure_delay_in_minutes'] - D['delay_recovered']).round().astype(int)
            code_ = lambda c: D[c].cat.codes.astype(int) if hasattr(D[c], 'cat') else D[c].astype('category').cat.codes.astype(int)
            D['key_cls_trv_gen'] = code_('class') * 100 + code_('type_of_travel') * 10 + code_('gender')
            D['key_fd_time'] = D['flight_distance'].astype(int) * 10 + D['departure/arrival_time_convenient'].astype(int)
        twins.update({'dep_delay_cat': 'departure_delay_in_minutes', 'arr_delay_cat': 'arr_delay_int'})
        te_src += ['key_cls_trv_gen', 'key_fd_time']
    return X, Xt, twins, te_src

def make_runner(variant):
    X, Xt, twins, te_src = build(variant)
    FULL = pd.concat([X, Xt], ignore_index=True)
    levels = {t: sorted(FULL[s].astype(int).unique()) for t, s in twins.items()}
    keys = [c for c in X.columns if c.startswith('key_')]
    drop = ['orig_prob'] + keys + (['arr_delay_int'] if 'B' in variant else [])
    cat_cols = ['gender', 'customer_type', 'type_of_travel', 'class'] + list(twins)
    def nn_frame(D):
        D = D.copy()
        for t, s in twins.items(): D[t] = pd.Categorical(D[s].astype(int), categories=levels[t])   # twins first: arr_delay_int is a twin source
        D = D.drop(columns=drop)
        assert not D.isna().any().any()
        return D
    # In-fold target encoding: fitted on the fold's TRAINING part only (inner cv=5), applied to validation/test.
    def add_te(Xtr, ytr, others):
        te = TargetEncoder(target_type='binary', cv=5, shuffle=True, random_state=42)
        cols = [f'te_{c}' for c in te_src]
        tr_te = pd.DataFrame(te.fit_transform(Xtr[te_src].astype(int), ytr), columns=cols, index=Xtr.index).astype('float32')
        outs = [pd.DataFrame(te.transform(o[te_src].astype(int)), columns=cols, index=o.index).astype('float32') for o in others]
        return [pd.concat([D, T], axis=1) for D, T in zip([Xtr] + list(others), [tr_te] + outs)]
    def fit_predict(tr_idx, va_idx, seed=SEED):
        Xtr, Xva, Xte = add_te(X.iloc[tr_idx], y[tr_idx], [X.iloc[va_idx], Xt])
        Xtr, Xva, Xte = nn_frame(Xtr), nn_frame(Xva), nn_frame(Xte)
        print(f'  variant {variant}: input columns {Xtr.shape[1]}, categorical {len(cat_cols)}, TE sources {len(te_src)}', flush=True)
        model = RealMLP_TD_Classifier(**{**PARAMS, 'random_state': seed})
        model.fit(Xtr, y[tr_idx], X_val=Xva, y_val=y[va_idx], cat_col_names=cat_cols)
        out = model.predict_proba(Xva)[:, 1], model.predict_proba(Xte)[:, 1]
        del model; gc.collect(); torch.cuda.empty_cache() if DEVICE == 'cuda' else None
        return out
    return fit_predict

PARAMS = dict(n_epochs=4, n_ens=N_ENS, batch_size=256, use_early_stopping=False, val_metric_name='1-auc_ovr', device=DEVICE, verbosity=0,
    lr=0.053, wd=0.0236, sq_mom=0.988, lr_sched='lin_cos_log_15', wd_sched='cos_log_15',
    first_layer_lr_factor=0.25, embedding_size=5, max_one_hot_cat_size=18,
    hidden_sizes=[512, 256, 128], act='silu', p_drop=0.05, p_drop_sched='expm4t',
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name='gelu', plr_lr_factor=0.1151, plr_sigma=2.33,
    use_ls=True, ls_eps=0.01, ls_eps_sched='sqrt_cos', add_front_scale=False, bias_init_mode='neg-uniform-dynamic-2',
    tfms=['one_hot', 'median_center', 'robust_scale', 'smooth_clip', 'embedding', 'l2_normalize'])

## 2. Fold 0 — three variants (resumable)

In [3]:
P = {}
for v in VARIANTS:
    d = PREDS_DIR / f'realmlp_rich_teach{v}_folds'; fo, ft = d / 'oof_0.npy', d / 'test_0.npy'
    if fo.exists() and ft.exists():
        P[v] = np.load(fo); print(f'variant {v}: loaded'); continue
    t0 = time.time(); p_va, p_te = make_runner(v)(tr0, va0)
    d.mkdir(exist_ok=True); np.save(fo, p_va); np.save(ft, p_te); P[v] = p_va
    print(f'variant {v}: {(time.time()-t0)/60:.1f} min', flush=True)
a_ref = roc_auc_score(y[va0], np.load(PREDS_DIR / 'realmlp_rich_refgpu_folds' / 'oof_0.npy'))
res = pd.DataFrame([{'run': 'reference (12)', 'fold 0 AUC': a_ref}] + [{'run': v, 'fold 0 AUC': roc_auc_score(y[va0], P[v])} for v in VARIANTS])
res['diff'] = res['fold 0 AUC'] - a_ref
print(res.round(5).to_string(index=False)); print('acceptance: diff >= +0.0002')

variant A: loaded


  variant B: input columns 114, categorical 21, TE sources 9


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


variant AB: 1.6 min


           run  fold 0 AUC    diff
reference (12)     0.96138 0.00000
             A     0.96138 0.00000
             B     0.96139 0.00002
            AB     0.96147 0.00009
acceptance: diff >= +0.0002


## 3. Mini-stack on the rows of fold 0
Same procedure as `15`: LR (C = 10) on the logits of the 10 members, inner 5-fold CV over the fold-0 rows; then + each variant, and + the two teacher logits as direct members.

In [4]:
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
as_logit = lambda o: lg(o) if (o.min() >= 0 and o.max() <= 1) else o
NAMES = {'cat_v1': 'cat_v1', 'cat_v2': 'cat_v2', 'xgb': 'xgb', 'lgbm_tuned': 'lgbm_tuned_final', 'mlp': 'mlp', 'realmlp': 'realmlp',
         'realmlp_rich': 'realmlp_rich', 'tabpfn': 'tabpfn_avg', 'tabpfn_raw': 'tabpfn_raw_avg', 'tabicl': 'tabicl'}
for k_, avg in [('realmlp', 'realmlp_avg'), ('realmlp_rich', 'realmlp_rich_avg')]:
    if (PREDS_DIR / f'oof_{avg}.npy').exists(): NAMES[k_] = avg
B = np.column_stack([as_logit(np.load(PREDS_DIR / f'oof_{s}.npy').astype(float))[va0] for s in NAMES.values()]); yv = y[va0]
EXTRA = {f'+ RealMLP {v}': lg(P[v])[:, None] for v in VARIANTS}
EXTRA['+ teacher logits (2)'] = TCH.values[va0].astype(float)
inner = list(StratifiedKFold(5, shuffle=True, random_state=42).split(B, yv))
def score(M):
    return [roc_auc_score(yv[b], M[b] @ LogisticRegression(C=10.0, max_iter=500).fit(M[a], yv[a]).coef_[0]) for a, b in inner]
base = np.array(score(B)); rows = [{'stack': '10 members', 'mean AUC': base.mean(), 'diff': 0.0, 'positive inner folds': '-'}]
for name, E in EXTRA.items():
    s = np.array(score(np.hstack([B, E]))); d = s - base
    rows.append({'stack': name, 'mean AUC': s.mean(), 'diff': d.mean(), 'positive inner folds': f'{(d > 0).sum()}/5'})
print(pd.DataFrame(rows).round(5).to_string(index=False))

               stack  mean AUC     diff positive inner folds
          10 members   0.96193  0.00000                    -
         + RealMLP A   0.96192 -0.00000                  3/5
         + RealMLP B   0.96193  0.00001                  4/5
        + RealMLP AB   0.96196  0.00003                  3/5
+ teacher logits (2)   0.96193  0.00001                  3/5


## Results & decisions

**Fold 0, RealMLP-rich, 4 networks, seed 0, GPU (paired reference 0.96138)**

| Variant | Fold 0 AUC | Difference | Mini-stack (10 members = 0.96193) | Positive inner folds |
|---|---|---|---|---|
| A: + 2 teacher logits | 0.96138 | +0.00000 | −0.00000 | 3/5 |
| B: + delay twins, TE of Class×Travel×Gender and Distance×Time | 0.96139 | +0.00002 | +0.00001 | 4/5 |
| A+B | 0.96147 | +0.00009 | +0.00003 | 3/5 |
| teacher logits as direct stack members | – | – | +0.00001 | 3/5 |

**Decision: rejected, step closed.** No variant reaches +0.0002 on fold 0; A+B's +0.00009 is within the seed spread of this model on fold 0 (0.00007 between three seeds) and its mini-stack gain is +0.00003 with mixed signs. Since neither the aux features (`13`–`14`) nor the teachers (here) helped the network, testing these new feature types in tree models is **skipped**.

**What this says about the data:** four different kinds of additions (recipe, in-row aux features, TabM, teachers / new keys) all land within ±0.0001 of the reference on fold 0. RealMLP-rich and the 10-model stack are at the ceiling of what these inputs carry; the original survey's signal is already used by the existing captains.


## Glossary
* **Teacher logit** — prediction of a model trained only on the original survey (`16`).
* **Delay twin** — a delay value given to the network also as a category.
* **Mini-stack** — logistic regression on members' logits, scored by an inner CV on the fold-0 rows.